# Stage 9 - MRL Eye Open/Closed CNN Baselines

This notebook trains the **MRL Eye eye-state specialist**. It is not a full drowsiness classifier.

The module outputs per-frame probabilities:

- `p_eye_closed`
- `p_eye_open`

These outputs are intended for later temporal smoothing or PERCLOS-like fusion with the completed YawDD mouth/yawn module.

## Notebook Notes

- Use a GPU runtime for full training.
- This notebook uses the existing Stage 8 subject-level split manifest.
- It does not rebuild the dataset and does not create image-level random splits.
- The `DATA_ROOT` variable controls path remapping when the manifest contains local absolute paths from another machine.

In [ ]:
# 1. Runtime and repository configuration
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/Drowsiness_Detection_Colab")
DRIVE_PROJECT_ROOT = DRIVE_ROOT

REPO_URL = "https://github.com/JohnCoffey-commits/Drowsiness-Detection.git"
REPO_BRANCH = "main"
REPO_DIR = DRIVE_ROOT / "repo"

DRIVE_DATA_DIR = DRIVE_ROOT / "data"
DRIVE_ZIP = DRIVE_DATA_DIR / "mrlEyes_2018_01.zip"

LOCAL_DATA_DIR = Path("/content/data")
LOCAL_ZIP = LOCAL_DATA_DIR / "mrlEyes_2018_01.zip"

# The manifest path is relative to the repo after os.chdir(REPO_DIR).
MANIFEST = Path("artifacts/mappings/mrl_eye_trainable_with_split.csv")
OUTPUT_DIR = DRIVE_ROOT / "outputs" / "mrl_eye"
DEBUG_OUTPUT_DIR = DRIVE_ROOT / "outputs" / "mrl_eye_debug"

FORCE_RECLONE_REPO = False

MODELS = ["resnet18", "mobilenet_v2", "efficientnet_b0"]
EPOCHS = 10
BATCH_SIZE = 64
IMAGE_SIZE = 224
NUM_WORKERS = 2
SEED = 42


In [ ]:
# 2. Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")
print("Drive project root:", DRIVE_PROJECT_ROOT)

## Prepare MRL Eye dataset on local Colab disk

The MRL Eye zip stays in Google Drive at `data/mrlEyes_2018_01.zip`.

For each Colab runtime, the notebook copies the zip to `/content/data`, extracts the dataset locally, and uses the local extracted folder as `DATA_ROOT`. This is faster and avoids relying on a partial Google Drive extraction. `/content` is temporary and will be lost after the runtime disconnects.

Training outputs are still saved back to Google Drive under `outputs/mrl_eye/`.


In [ ]:
# 3. Copy MRL Eye zip from Google Drive to local Colab disk
from pathlib import Path
import shutil
import time
import zipfile

DRIVE_ROOT = Path("/content/drive/MyDrive/Drowsiness_Detection_Colab")
DRIVE_PROJECT_ROOT = DRIVE_ROOT
DRIVE_DATA_DIR = DRIVE_ROOT / "data"
DRIVE_ZIP = DRIVE_DATA_DIR / "mrlEyes_2018_01.zip"

LOCAL_DATA_DIR = Path("/content/data")
LOCAL_ZIP = LOCAL_DATA_DIR / "mrlEyes_2018_01.zip"

OUTPUT_DIR = DRIVE_ROOT / "outputs" / "mrl_eye"
DEBUG_OUTPUT_DIR = DRIVE_ROOT / "outputs" / "mrl_eye_debug"

LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DRIVE_ZIP:", DRIVE_ZIP)
print("DRIVE_ZIP exists:", DRIVE_ZIP.exists())

if not DRIVE_ZIP.exists():
    raise FileNotFoundError(f"MRL Eye zip not found in Google Drive: {DRIVE_ZIP}")

if not LOCAL_ZIP.exists():
    print("Copying MRL Eye zip from Google Drive to local /content/data ...")
    start = time.time()
    shutil.copy2(DRIVE_ZIP, LOCAL_ZIP)
    print(f"Copy complete in {(time.time() - start) / 60:.2f} minutes")
else:
    print("Local zip already exists. Skipping copy.")

print("LOCAL_ZIP:", LOCAL_ZIP)
print("LOCAL_ZIP size GB:", LOCAL_ZIP.stat().st_size / 1024**3)

with zipfile.ZipFile(LOCAL_ZIP, "r") as z:
    names = z.namelist()
    pngs = [name for name in names if name.lower().endswith(".png")]
    print("PNG files inside zip:", len(pngs))
    s0011_pngs = [name for name in pngs if "/s0011/" in name or name.startswith("s0011/")]
    print("s0011 PNG files inside zip:", len(s0011_pngs))


In [ ]:
# 4. Extract locally and validate completeness
EXPECTED_TOTAL_IMAGES = 84898


def find_mrl_eye_root(base_dir: Path):
    candidates = [
        base_dir / "mrlEyes_2018_01",
        base_dir / "dataset" / "mrlEyes_2018_01",
        base_dir / "mrlEyes_2018_01" / "mrlEyes_2018_01",
    ]
    for candidate in candidates:
        if (
            (candidate / "annotation.txt").exists()
            and (candidate / "s0001").exists()
            and (candidate / "s0037").exists()
        ):
            return candidate
    return None


def count_pngs(path: Path) -> int:
    if path is None or not path.exists():
        return 0
    return sum(1 for _ in path.rglob("*.png"))


def subject_png_count(root: Path, subject: str) -> int:
    if root is None:
        return 0
    folder = root / subject
    if not folder.exists():
        return 0
    return sum(1 for _ in folder.glob("*.png"))


def is_complete_mrl_eye_root(root: Path) -> bool:
    if root is None or not root.exists():
        print("MRL Eye root not found.")
        return False
    total_png = count_pngs(root)
    checks = {
        "annotation.txt": (root / "annotation.txt").exists(),
        "s0001_pngs": subject_png_count(root, "s0001"),
        "s0011_pngs": subject_png_count(root, "s0011"),
        "s0037_pngs": subject_png_count(root, "s0037"),
        "total_png": total_png,
    }
    print("MRL Eye completeness checks:", checks)
    return (
        checks["annotation.txt"]
        and checks["s0001_pngs"] > 0
        and checks["s0011_pngs"] > 0
        and checks["s0037_pngs"] > 0
        and checks["total_png"] == EXPECTED_TOTAL_IMAGES
    )


DATA_ROOT = find_mrl_eye_root(LOCAL_DATA_DIR)

if not is_complete_mrl_eye_root(DATA_ROOT):
    print("Local MRL Eye extraction is missing or incomplete. Re-extracting locally...")
    for candidate in [
        LOCAL_DATA_DIR / "mrlEyes_2018_01",
        LOCAL_DATA_DIR / "dataset",
    ]:
        if candidate.exists():
            print(f"Removing incomplete local folder only: {candidate}")
            shutil.rmtree(candidate)

    start = time.time()
    with zipfile.ZipFile(LOCAL_ZIP, "r") as zip_ref:
        zip_ref.extractall(LOCAL_DATA_DIR)
    print(f"Extraction complete in {(time.time() - start) / 60:.2f} minutes")

DATA_ROOT = find_mrl_eye_root(LOCAL_DATA_DIR)

if not is_complete_mrl_eye_root(DATA_ROOT):
    raise RuntimeError(
        "Local MRL Eye extraction is still incomplete after re-extraction. "
        "Check the zip file integrity."
    )

DATA_ROOT_STR = str(DATA_ROOT)
OUTPUT_DIR_STR = str(OUTPUT_DIR)
DEBUG_OUTPUT_DIR_STR = str(DEBUG_OUTPUT_DIR)

print("Final DATA_ROOT:", DATA_ROOT)
print("DATA_ROOT_STR:", DATA_ROOT_STR)
print("OUTPUT_DIR_STR:", OUTPUT_DIR_STR)
print("DEBUG_OUTPUT_DIR_STR:", DEBUG_OUTPUT_DIR_STR)


In [ ]:
# 3. Clone and prepare repository code
import os
import shutil
import subprocess
import sys

if FORCE_RECLONE_REPO and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Repository already exists at {REPO_DIR}. Reusing it.")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Working directory:", Path.cwd())

In [ ]:
# 4. Install/import dependencies if needed
import importlib
import subprocess
import sys

for package, import_name in [("pandas", "pandas"), ("scikit-learn", "sklearn"), ("matplotlib", "matplotlib"), ("tqdm", "tqdm")]:
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import torch
import torchvision
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Path Setup

`MANIFEST` points to the Stage 8 split manifest inside the cloned repository.

`DATA_ROOT` is resolved by the local extraction cells above. It should point to the `/content/data/...` MRL Eye folder containing `annotation.txt`, subject folders, and exactly `84,898` PNG files. The training script receives `--data-root DATA_ROOT_STR` so it does not use the old broken Google Drive extraction.


In [ ]:
# 7. Verify required training inputs
required_paths = {
    "Resolved DATA_ROOT": Path(DATA_ROOT_STR),
    "Stage 8 manifest": MANIFEST,
}

missing = []
for name, path in required_paths.items():
    exists = path.exists()
    print(f"{name}: {path} -> {'OK' if exists else 'MISSING'}")
    if not exists:
        missing.append(f"{name}: {path}")

if missing:
    raise FileNotFoundError("Missing required inputs:\n" + "\n".join(missing))


In [ ]:
# 8. Small manifest sanity check
import pandas as pd
from pathlib import Path

df = pd.read_csv(MANIFEST, dtype={"subject_id": str, "sensor_id": str})
print("Rows:", len(df))
print("Columns:", list(df.columns))
print("Labels:", df[["label", "label_name"]].drop_duplicates().sort_values("label").to_string(index=False))
print("Split/class counts:")
print(df.groupby(["split", "label_name"]).size().unstack(fill_value=0))

assert set(df["label"].astype(int)) == {0, 1}
assert set(df["split"]) == {"train", "val", "test"}
assert (df.groupby("subject_id")["split"].nunique() == 1).all(), "Subject leakage detected"


In [ ]:
# 9. Smoke test: ResNet18 only, tiny subset
smoke_cmd = [
    sys.executable, "src/training/train_mrl_eye_baselines.py",
    "--models", "resnet18",
    "--epochs", "1",
    "--batch-size", "16",
    "--max-samples-per-split", "128",
    "--manifest", str(MANIFEST),
    "--data-root", DATA_ROOT_STR,
    "--output-dir", DEBUG_OUTPUT_DIR_STR,
    "--require-pretrained",
]
print("Running:", " ".join(smoke_cmd))
subprocess.run(smoke_cmd, check=True)


## Full Training

Run full training only after:

- local extraction is complete
- total PNG count is 84,898
- smoke test completes successfully


In [ ]:
# 10. Launch full Stage 9 training
train_cmd = [
    sys.executable, "src/training/train_mrl_eye_baselines.py",
    "--models", *MODELS,
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--image-size", str(IMAGE_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--seed", str(SEED),
    "--manifest", str(MANIFEST),
    "--data-root", DATA_ROOT_STR,
    "--output-dir", OUTPUT_DIR_STR,
    "--require-pretrained",
]
print("Running:", " ".join(train_cmd))
subprocess.run(train_cmd, check=True)


In [ ]:
# 11. Display final metrics table
results_csv = Path(OUTPUT_DIR_STR) / "results/mrl_eye_initial_results.csv"
results = pd.read_csv(results_csv)
display_cols = [
    "model", "best_val_macro_f1", "test_accuracy", "test_macro_f1",
    "test_recall_closed", "test_false_open_count", "test_false_closed_count",
]
optional_cols = [
    "selected_threshold_from_val", "test_selected_threshold_macro_f1",
    "test_selected_threshold_recall_closed", "test_selected_threshold_false_open_count",
]
display(results[[col for col in display_cols + optional_cols if col in results.columns]])


In [ ]:
# 12. Display confusion matrices and training curves
from IPython.display import Image as IPyImage, display

for model_name in MODELS:
    print("\n", model_name)
    curve = Path(OUTPUT_DIR_STR) / "figures" / f"{model_name}_training_curve.png"
    cm = Path(OUTPUT_DIR_STR) / "figures" / f"{model_name}_confusion_matrix.png"
    if curve.exists():
        display(IPyImage(filename=str(curve)))
    if cm.exists():
        display(IPyImage(filename=str(cm)))


In [ ]:
# 13. Inspect threshold sweeps for p_eye_closed
for model_name in MODELS:
    for split_name in ["val", "test"]:
        sweep_path = Path(OUTPUT_DIR_STR) / "results" / f"{model_name}_{split_name}_threshold_sweep.csv"
        if sweep_path.exists():
            print("\n", model_name, split_name)
            display(pd.read_csv(sweep_path))


## Interpretation Notes

- Prioritize validation macro F1 for checkpoint selection.
- Review closed-eye recall carefully. False-open errors are true closed-eye frames predicted as open, which can hide eye-closure events.
- Do not choose a threshold that predicts nearly everything as closed. Use the threshold sweep to balance closed recall with macro F1 and false-closed errors.
- The next stage should use `p_eye_closed` over time for smoothing or PERCLOS-like logic, then fuse with YawDD mouth/yawn outputs.